# ML from Scratch - 06a - Decision Tree Classifier

## Importing

In [ ]:
import importlib
import my_decision_tree_classifier

importlib.reload(my_decision_tree_classifier)
from my_decision_tree_classifier import Decision_Tree_Classifier

## Synthetic Data

In [ ]:
import numpy as np

np.random.seed(42)

X = np.random.randn(100, 5)

weights = np.random.randn(X.shape[1])

proba = X @ weights.T + 0.5 + np.random.randn(X.shape[0])

sigmoid = 1 / (1 + (np.exp(-proba)))

y = np.where(sigmoid >= 0.5, 1, 0)

y

### Using Gini

In [ ]:
model_gini = Decision_Tree_Classifier(mode='gini')

model_gini.fit(X, y)

In [ ]:
from sklearn.metrics import accuracy_score

y_pred_gini = model_gini.predict(X)

accuracy_score(y, y_pred_gini)

### Using Entropy

In [ ]:
model_entropy = Decision_Tree_Classifier(mode='entropy')

model_entropy.fit(X, y)

In [ ]:
y_pred_entropy = model_entropy.predict(X)

accuracy_score(y, y_pred_entropy)

## Titanic

In [ ]:
import pandas as pd

train_df = pd.read_csv('../data/titanic/train.csv').set_index('PassengerId')

train_df.head()

### EDA

In [ ]:
train_df.isnull().sum()

In [ ]:
train_df.describe()

In [ ]:
train_df.info()

In [ ]:
train_df['Embarked'].unique()

In [ ]:
train_df['Age'].sort_values().unique()

### Preprocessing

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class FamilyFeatures(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        X = X.copy()
        X['FamilySize'] = X['Parch'] + X['SibSp'] + 1
        X['IsAlone'] = (X['FamilySize'] == 1).astype(int)

        return X

In [ ]:
from sklearn.pipeline import make_pipeline

feature_eng_pipeline = make_pipeline(
    FamilyFeatures(),
)

feature_eng_pipeline

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

num_pipeline = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler()
)

num_pipeline

In [ ]:
from sklearn.preprocessing import OneHotEncoder

obj_pipeline = make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OneHotEncoder(handle_unknown='ignore')
)

obj_pipeline

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

gender_pipeline = make_pipeline(
    OrdinalEncoder()
)

gender_pipeline

In [ ]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
        ('num', num_pipeline, ['Age', 'Fare', 'FamilySize', 'SibSp', 'Parch']),
        ('obj', obj_pipeline, ['Embarked']),
        ('gender', gender_pipeline, ['Sex']),
        ('pass', 'passthrough', ['Pclass', 'IsAlone'])
        ], remainder='drop')

preprocessor

In [ ]:
target = 'Survived'

X = train_df.drop(columns=[target])
y = train_df[target]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)

X_train.shape, X_val.shape

In [ ]:
X_train_engineered = feature_eng_pipeline.fit_transform(X_train)
X_val_engineered = feature_eng_pipeline.transform(X_val)
X_full_engineered = feature_eng_pipeline.transform(X)

X_train_engineered.shape, X_val_engineered.shape, X_full_engineered.shape

In [ ]:
X_train_preprocessed = preprocessor.fit_transform(X_train_engineered)
X_val_preprocessed = preprocessor.transform(X_val_engineered)
X_full_preprocessed = preprocessor.transform(X_full_engineered)

X_train_preprocessed.shape, X_val_preprocessed.shape, X_full_preprocessed.shape

In [ ]:
X_train_preprocessed = pd.DataFrame(X_train_preprocessed, index=X_train_engineered.index, columns=preprocessor.get_feature_names_out())
X_val_preprocessed = pd.DataFrame(X_val_preprocessed, index=X_val_engineered.index, columns=preprocessor.get_feature_names_out())
X_full_preprocessed = pd.DataFrame(X_full_preprocessed, index=X_full_engineered.index, columns=preprocessor.get_feature_names_out())

X_train_preprocessed.head()

### Model Training

#### My Model (Gini)

In [ ]:
model_gini.fit(X_train_preprocessed, y_train)

In [ ]:
y_pred_gini = model_gini.predict(X_val_preprocessed)

accuracy_score(y_val, y_pred_gini)

#### My Model (Entropy)

In [ ]:
model_entropy.fit(X_train_preprocessed, y_train)

In [ ]:
y_pred_entropy = model_entropy.predict(X_val_preprocessed)

accuracy_score(y_val, y_pred_entropy)

#### SciKit-Learn

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model_sklearn = DecisionTreeClassifier(random_state=42)

model_sklearn.fit(X_train_preprocessed, y_train)

In [ ]:
y_pred_sklearn = model_sklearn.predict(X_val_preprocessed)

accuracy_score(y_val, y_pred_sklearn)

### Retraining and Submission

In [ ]:
model_gini.fit(X_full_preprocessed, y)

In [ ]:
model_entropy.fit(X_full_preprocessed, y)

In [ ]:
model_sklearn.fit(X_full_preprocessed, y)

In [ ]:
test_df = pd.read_csv('../data/titanic/test.csv').set_index('PassengerId')

test_df.head()

In [ ]:
X_test_engineered = feature_eng_pipeline.transform(test_df)
X_test_preprocessed = preprocessor.transform(X_test_engineered)

X_test_preprocessed = pd.DataFrame(X_test_preprocessed, index=X_test_engineered.index, columns=preprocessor.get_feature_names_out())

X_test_preprocessed.head()

In [ ]:
y_test_gini = model_gini.predict(X_test_preprocessed)
y_test_entropy = model_entropy.predict(X_test_preprocessed)
y_test_sklearn = model_sklearn.predict(X_test_preprocessed)

In [ ]:
submission_gini_df = pd.DataFrame(y_test_gini, columns=['Survived'], index=test_df.index)

submission_gini_df.to_csv('../data/titanic/submissions/submission_tree_gini.csv', sep=',')

In [ ]:
submission_entropy_df = pd.DataFrame(y_test_entropy, columns=['Survived'], index=test_df.index)

submission_entropy_df.to_csv('../data/titanic/submissions/submission_tree_entropy.csv', sep=',')

In [ ]:
submission_sklearn_df = pd.DataFrame(y_test_sklearn, columns=['Survived'], index=test_df.index)

submission_sklearn_df.to_csv('../data/titanic/submissions/submission_tree_sklearn.csv', sep=',')

Kaggle Score for my model (Gini): ``

Kaggle Score for my model (Entropy): ``

Kaggle Score for SciKit-Learn model: ``